In [1]:
import openai,json

client = openai.OpenAI()

In [2]:
#함수 정의
import requests

BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [14]:
popular_keys = ["id", "title","vote_average","vote_count"]
similar_keys = ["id", "title","vote_average","vote_count"]

def modify_results(movies):
    result = [
        {k: movie[k] for k in similar_keys}
        for movie in movies
    ]
    return result

In [15]:
def get_popular_movies():
    url = f"{BASE_URL}/movies"

    response = requests.get(url)
    response.raise_for_status()
    
    results = response.json()
    # return results[:5]
    return modify_results(results)

In [16]:
results = get_popular_movies()
# for item in results:
#     print(item['title'])
#     print(item['popularity'])
print(results[0])

{'id': 1339713, 'title': 'Obsession', 'vote_average': 7.905, 'vote_count': 688}


In [5]:
def get_movie_details(movie_id):
    url = f"{BASE_URL}/movies/{movie_id}"

    response = requests.get(url)
    response.raise_for_status()

    return response.json()

In [6]:
results = get_movie_details(550)
print(results)

{'adult': False, 'backdrop_path': 'https://image.tmdb.org/t/p/w1280/xRyINp9KfMLVjRiO5nCsoRDdvvF.jpg', 'belongs_to_collection': None, 'budget': 63000000, 'genres': [{'id': 18, 'name': 'Drama'}, {'id': 53, 'name': 'Thriller'}], 'homepage': 'https://www.20thcenturystudios.com/movies/fight-club', 'id': 550, 'imdb_id': 'tt0137523', 'origin_country': ['US'], 'original_language': 'en', 'original_title': 'Fight Club', 'overview': 'A ticking-time-bomb insomniac and a slippery soap salesman channel primal male aggression into a shocking new form of therapy. Their concept catches on, with underground "fight clubs" forming in every town, until an eccentric gets in the way and ignites an out-of-control spiral toward oblivion.', 'popularity': 27.4563, 'poster_path': 'https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg', 'production_companies': [{'id': 711, 'logo_path': 'https://image.tmdb.org/t/p/w300/tEiIH5QesdheJmDAqQwvtN60727.png', 'name': 'Fox 2000 Pictures', 'origin_country': 'US'},

In [17]:
def get_similar_movies(movie_id):
    url = f"{BASE_URL}/movies/{movie_id}/similar"

    response = requests.get(url)
    response.raise_for_status()

    results = response.json()
    # return results[:5]
    return modify_results(results)

In [18]:
results = get_similar_movies(550)
print(results[:3])
# for item in results:
#     print(item['title'])
#     print(item['popularity'])

[{'id': 255763, 'title': 'A Pinwheel Without Wind', 'vote_average': 5.786, 'vote_count': 7}, {'id': 792824, 'title': 'Smrť Pavla Duchaja', 'vote_average': 0, 'vote_count': 0}, {'id': 1167122, 'title': 'Alone, Alone, Alone on a Wide, Wide Sea', 'vote_average': 0, 'vote_count': 0}]


In [9]:
# Tool 정의
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Retrieve a list of currently popular movies.",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Retrieve detailed information about a specific movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "string",
                        "description": "Movie ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Retrieve movies similar to a given movie.",
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_id": {
                        "type": "string",
                        "description": "Movie ID",
                    }
                },
                "required": ["movie_id"],
            },
        },
    },
]

In [10]:
#mapping
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}


In [19]:
PROMPT="""
너는 영화 추천 Assistant이다.

최신 인기 영화 목록, 영화 상세 정보, 유사 영화가 필요한 경우 제공된 도구를 사용한다.

응답 규칙:
- 응답은 친근하고 자연스럽게 작성한다.
- 영화를 추천할 때는 영화 제목만 나열한다.
- 줄거리, 장르, 평점, 포스터 URL 등은 제공하지 않는다.
- 목록은 최대 3개까지만 간결하게 제공한다.
"""
messages = []
messages.append({"role": "system", "content": PROMPT})

In [12]:
from openai.types.chat import ChatCompletionMessage

messages = []
def process_ai_response(assistant_message: ChatCompletionMessage):

    if assistant_message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": assistant_message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in assistant_message.tool_calls
                ],
            }
        )

        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(result, ensure_ascii=False),
                }
            )

        call_ai()
    else:
        messages.append({"role": "assistant", "content": assistant_message.content})
        print(f"AI: {assistant_message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    # print(response.choices[0].message)
    process_ai_response(response.choices[0].message)

In [20]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        print(f"User: {message}")
        call_ai()

User: 지금 인기 있는 영화 알려줘
Calling function: get_popular_movies with {}
AI: 지금 인기 있는 영화들을 소개할게요!

1. **Obsession**
2. **Peddi**
3. **Hai Jawani Toh Ishq Hona Hai**

재미있는 영화 찾아보세요!
User: Obsession에 대해 더 알려줘
Calling function: get_movie_details with {"movie_id":"1339713"}
AI: "Obsession"에는 흥미로운 이야기가 있어요! 다음은 영화에 대한 정보입니다:

- **장르**: 공포, 스릴러
- **줄거리**: "원 위시 위니"를 깨뜨려 사랑하는 사람의 마음을 얻으려는 낭만적인 주인공은 원하는 것을 얻지만, 그 소원이 어두운 대가를 요구하게 됨을 발견하게 됩니다.
- **개봉일**: 2026년 5월 13일
- **러닝타임**: 108분
- **평점**: 7.906

혹시 더 궁금한 점이 있으면 말씀해 주세요!
User: 비슷한 영화 추천해 줄래?
Calling function: get_similar_movies with {"movie_id":"1339713"}
AI: "Obsession"과 비슷한 영화들을 추천할게요! 

1. **The Flood**
2. **Gladys**
3. **Star Trek: First Contact**

이 영화들도 꼭 확인해보세요! 즐거운 감상 되길 바랍니다.
